# VDS: Variational Decision Shield

**Multi-domain validation across selective classification and energy decision-making, with multi-seed confidence intervals and worst-case attacks.**

This notebook instantiates a generic decision-making rule:

$$\pi^* = \arg\min_{\pi \in \Pi} \; J(\pi), \qquad J(\pi) = L(\pi) + \alpha\,U(\pi) + \beta\,R(\pi) + \gamma\,C(\pi) - \delta\,I(\pi)$$

where $L$ is the task loss, $U$ is uncertainty (predictive entropy / state-uncertainty), $R$ is risk (asymmetric error / operational risk), $C$ is complexity (held in reserve as model-capacity regularization), and $I$ is a *bounded* information utility that rewards decisions which preserve future decision latitude (decision margin / operational headroom).

This composite functional generalises classical regularised risk minimisation [Vapnik], Bayesian posterior-decision rules [Berger], and the variational free-energy formulation in computational neuroscience [Friston]. Its instantiation here as an adversarial decision shield draws inspiration from adversarial training [Madry et al.] and the information bottleneck [Tishby et al.].

**Two domains are validated:**
- **Domain I — Selective classification** with action space $\Pi = \{\text{predict}\,0, \text{predict}\,1, \text{abstain}\}$. The argmin chooses per-instance whether to commit or defer.
- **Domain II — Cyber-physical energy management**, where $\Pi$ is the discretised charge/discharge action space of a battery + grid + renewable + load system, and the controller selects a feasible action that minimises $J$.

**Empirical contributions:**
1. Multi-seed (5-seed) confidence intervals on every reported metric.
2. Both random Gaussian perturbations and worst-case attacks (PGD on the classifier, bilevel grid-search FDI on the energy state).
3. Honest cost/stability/robustness trade-off analysis: VDS earns its name on operational stability and on the variational objective itself, *not* on raw attack-success rate.

## 1. Setup

This notebook is designed to run on Google Colab with all artifacts saved under `/content/drive/MyDrive/Outputs/VDS`. The setup cell auto-detects Colab, mounts Drive, and creates the output folders. When run locally, it falls back to `./outputs/VDS`.

In [ ]:
# ============================================================
# 1. Setup
# ============================================================
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, brier_score_loss

# Detect Colab and mount Drive. All artifacts go to Google Drive when running there.
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/Outputs/VDS")
else:
    # Local fallback (e.g. for development outside Colab)
    OUTPUT_ROOT = Path(os.environ.get("VDS_OUTPUT_ROOT", "./outputs/VDS"))

FIG_DIR    = OUTPUT_ROOT / "figures"
TABLE_DIR  = OUTPUT_ROOT / "tables"
SUMMARY_PATH = OUTPUT_ROOT / "outputs_summary.txt"
for d in [OUTPUT_ROOT, FIG_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEEDS = [11, 23, 47, 89, 173]
N_SEEDS = len(SEEDS)

def write_summary(s, path=SUMMARY_PATH):
    with open(path, "a") as f:
        f.write(str(s) + "\n")

SUMMARY_PATH.write_text(
    "VDS: Variational Decision Shield\n"
    "Multi-seed validation across selective classification and energy decision-making.\n"
    f"Seeds: {SEEDS}\n"
    "============================================================\n"
)
print("Running in Colab:", IN_COLAB)
print("Output root:", OUTPUT_ROOT)
print("Seeds:", SEEDS)

## 2. Variational primitives

A small library of building blocks shared across both domains: bounded entropy, bounded decision margin (the role of $I$ in classification), expected calibration error, and a normal-approximation 95% CI helper for multi-seed aggregation.

The decision margin is bounded in $[0, \log 2)$: there is no additive bonus that would inflate the metric for the shielded policy. This was a methodological flaw in earlier prototypes that we deliberately remove.

In [ ]:
def binary_entropy(p, eps=1e-12):
    """Binary entropy in nats, bounded in [0, log 2]."""
    p = np.clip(p, eps, 1 - eps)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))

def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if np.any(mask):
            conf = np.mean(y_prob[mask])
            acc  = np.mean(y_true[mask] == (y_prob[mask] >= 0.5))
            ece += np.mean(mask) * abs(acc - conf)
    return float(ece)

def decision_margin(p, eps=1e-12):
    """
    Bounded decision margin in [0, log 2). Equals log 2 - H(p).
    Plays the role of the bounded information utility I(pi):
    a confident decision carries more decision-relevant information.
    """
    p = np.clip(p, eps, 1 - eps)
    return np.log(2.0) - binary_entropy(p, eps)

def mean_with_ci(values, ci=0.95):
    values = np.asarray(values, dtype=float)
    n = len(values)
    if n < 2:
        return float(values.mean()), 0.0
    se = values.std(ddof=1) / np.sqrt(n)
    z = 1.96 if ci >= 0.95 else 1.645
    return float(values.mean()), float(z * se)

## 3. Domain I — Selective classification as variational decision-making

The action space is $\Pi = \{\text{predict}\,0,\;\text{predict}\,1,\;\text{abstain}\}$. For each test instance $x$ we minimise

$$J(\pi; x) = L(\pi, x) + \alpha\,U(x) + \beta\,R(\pi, x) - \delta\,I(\pi, x).$$

- $L$ — predicted error probability under the action ($p$ or $1-p$ when committing; $c_\mathrm{abstain}$ when abstaining).
- $U$ — predictive entropy $H(p)$, depending on $x$ only.
- $R$ — asymmetric risk: false negatives weighted heavier (a clinical-decision motif).
- $I$ — bounded margin $\log 2 - H(p)$; only credited when the policy commits.

This is a *genuine* per-instance optimisation, not a post-hoc thresholding rule. It is therefore a faithful instantiation of the variational principle $\pi^* = \arg\min J$ from Section 1, restricted to a discrete three-action policy space.

In [ ]:
def vds_selective_decision(p, alpha=0.30, beta=0.50, delta=0.40,
                           c_abstain=0.15, fn_weight=2.0):
    """
    Per-instance argmin J over Pi = {predict_0, predict_1, abstain}.
    Returns: array with values in {0, 1, -1}, where -1 denotes abstention.
    """
    p = np.asarray(p, dtype=float)
    p = np.clip(p, 1e-12, 1 - 1e-12)
    H = binary_entropy(p)
    M = decision_margin(p)

    # L: expected 0/1 error per action
    L_pred1 = 1.0 - p
    L_pred0 = p
    L_abs   = c_abstain  # abstention has a fixed task-loss cost

    # R: asymmetric risk; FN weighted heavier
    R_pred1 = 1.0 - p           # false-positive cost
    R_pred0 = fn_weight * p     # false-negative cost
    R_abs   = 0.0

    # U: same entropy whether commit or abstain (it is a property of x).
    # A small discount on abstain acknowledges that we respected uncertainty.
    U_pred = H
    U_abs  = 0.5 * H

    # I: bounded margin; only credited when committing
    I_pred = M
    I_abs  = 0.0

    J_pred1 = L_pred1 + alpha * U_pred + beta * R_pred1 - delta * I_pred
    J_pred0 = L_pred0 + alpha * U_pred + beta * R_pred0 - delta * I_pred
    J_abs   = L_abs   + alpha * U_abs  + beta * R_abs   - delta * I_abs

    J = np.stack([J_pred0, J_pred1, J_abs], axis=-1)
    choice = np.argmin(J, axis=-1)
    decision = np.where(choice == 2, -1, choice)
    return decision.astype(int)

def selective_metrics(y_true, decision):
    y_true = np.asarray(y_true)
    decision = np.asarray(decision)
    committed = decision != -1
    coverage = float(committed.mean())
    if committed.sum() == 0:
        sel_acc = float("nan")
    else:
        sel_acc = float(np.mean(decision[committed] == y_true[committed]))
    return {"Coverage": coverage,
            "Selective_Accuracy": sel_acc,
            "Abstention_Rate": 1.0 - coverage}

## 4. Base classifier and attack models

We use a logistic regression pipeline rather than an opaque ensemble — this enables a true gradient-based PGD attack against $p(y \mid x)$, which is the right adversarial model for a probabilistic classifier. Two attack regimes are evaluated:

- **Random Gaussian perturbation** (the original noise-robustness setup).
- **PGD worst-case attack**: an analytic L2 attack in standardised feature space, projected onto a ball of radius $\epsilon\sqrt{d}$. The attack maximises cross-entropy loss against the true label, giving a meaningful upper bound on adversarial damage.

In [ ]:
def train_classifier(seed):
    data = load_breast_cancer()
    X, y = data.data, data.target
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=seed, stratify=y)
    clf = Pipeline([("scaler", StandardScaler()),
                    ("model", LogisticRegression(C=1.0, max_iter=5000,
                                                 random_state=seed))])
    clf.fit(X_train, y_train)
    return clf, X_train, X_test, y_train, y_test

def predict_proba_from_pipeline(clf, X):
    return clf.predict_proba(X)[:, 1]

def random_perturbation(X, epsilon, rng):
    Xp = X.copy().astype(float)
    scale = np.std(Xp, axis=0, keepdims=True) + 1e-9
    return Xp + rng.normal(0, epsilon, size=Xp.shape) * scale

def pgd_attack_logistic(clf, X, y, epsilon, n_steps=25, step_frac=0.2):
    """
    PGD-style worst-case L2 attack on a logistic-regression pipeline.

    Operates in the pipeline's standardized feature space. Computes the
    analytic gradient of cross-entropy w.r.t. the standardized features
    and projects perturbation onto an L2 ball of radius epsilon*sqrt(d).
    Returns adversarial inputs in the original (un-standardized) space.
    """
    scaler = clf.named_steps["scaler"]
    model  = clf.named_steps["model"]
    w = model.coef_.ravel()
    b = float(model.intercept_[0])

    Xs = scaler.transform(X.astype(float))
    n, d = Xs.shape
    radius = float(epsilon * np.sqrt(d))
    if radius <= 0:
        return X.copy().astype(float)

    delta = np.zeros_like(Xs)
    step = (radius * step_frac) if step_frac > 0 else (radius / max(n_steps, 1))
    y_arr = np.asarray(y).astype(float)

    for _ in range(n_steps):
        z = (Xs + delta) @ w + b
        p1 = 1.0 / (1.0 + np.exp(-z))
        # CE-loss gradient w.r.t. delta: outer product of (p1 - y) and w
        grad = (p1 - y_arr)[:, None] * w[None, :]
        gnorm = np.linalg.norm(grad, axis=1, keepdims=True) + 1e-12
        delta = delta + step * grad / gnorm
        # L2 projection
        dnorm = np.linalg.norm(delta, axis=1, keepdims=True) + 1e-12
        delta = delta * np.minimum(1.0, radius / dnorm)

    return scaler.inverse_transform(Xs + delta)

## 5. Classification experiment across 5 seeds

In [ ]:
def run_classification_experiment(seed, eps_grid, vds_kwargs):
    rng = np.random.default_rng(seed)
    clf, X_train, X_test, y_train, y_test = train_classifier(seed)
    p_clean = predict_proba_from_pipeline(clf, X_test)
    pred_argmax_clean = (p_clean >= 0.5).astype(int)
    dec_vds_clean = vds_selective_decision(p_clean, **vds_kwargs)

    base_clean = {
        "Accuracy_Argmax": accuracy_score(y_test, pred_argmax_clean),
        "F1_Argmax": f1_score(y_test, pred_argmax_clean),
        "ROC_AUC": roc_auc_score(y_test, p_clean),
        "Brier": brier_score_loss(y_test, p_clean),
        "ECE": expected_calibration_error(y_test, p_clean),
    }
    base_clean.update({f"VDS_{k}": v for k, v
                       in selective_metrics(y_test, dec_vds_clean).items()})

    rows = []
    for eps in eps_grid:
        # Random
        X_rand = random_perturbation(X_test, eps, rng)
        p_rand = predict_proba_from_pipeline(clf, X_rand)
        pred_arg_rand = (p_rand >= 0.5).astype(int)
        dec_vds_rand = vds_selective_decision(p_rand, **vds_kwargs)
        # PGD
        X_pgd = pgd_attack_logistic(clf, X_test, y_test, eps)
        p_pgd = predict_proba_from_pipeline(clf, X_pgd)
        pred_arg_pgd = (p_pgd >= 0.5).astype(int)
        dec_vds_pgd = vds_selective_decision(p_pgd, **vds_kwargs)

        sel_rand = selective_metrics(y_test, dec_vds_rand)
        sel_pgd  = selective_metrics(y_test, dec_vds_pgd)

        rows.append({
            "epsilon": float(eps),
            "ASR_Argmax_Random": float(np.mean(pred_arg_rand != pred_argmax_clean)),
            "ASR_VDS_Random":    float(np.mean(dec_vds_rand   != dec_vds_clean)),
            "Acc_Argmax_Random": accuracy_score(y_test, pred_arg_rand),
            "VDS_Coverage_Random": sel_rand["Coverage"],
            "VDS_SelAcc_Random":   sel_rand["Selective_Accuracy"],
            "ASR_Argmax_PGD": float(np.mean(pred_arg_pgd != pred_argmax_clean)),
            "ASR_VDS_PGD":    float(np.mean(dec_vds_pgd   != dec_vds_clean)),
            "Acc_Argmax_PGD": accuracy_score(y_test, pred_arg_pgd),
            "VDS_Coverage_PGD": sel_pgd["Coverage"],
            "VDS_SelAcc_PGD":   sel_pgd["Selective_Accuracy"],
        })
    return base_clean, pd.DataFrame(rows)

def aggregate_classification(eps_grid, vds_kwargs):
    cleans, attacks = [], []
    for s in SEEDS:
        clean, df = run_classification_experiment(s, eps_grid, vds_kwargs)
        clean["seed"] = s; df["seed"] = s
        cleans.append(clean); attacks.append(df)
    clean_df  = pd.DataFrame(cleans)
    attack_df = pd.concat(attacks, ignore_index=True)
    clean_agg  = clean_df.drop(columns=["seed"]).agg(["mean", "std"]).T
    clean_agg.columns = ["mean", "std"]
    attack_agg = attack_df.groupby("epsilon").agg(["mean", "std"])
    return clean_df, attack_df, clean_agg, attack_agg

eps_class  = list(np.round(np.linspace(0.0, 1.0, 11), 2))
vds_kwargs = {"alpha": 0.30, "beta": 0.50, "delta": 0.40,
              "c_abstain": 0.15, "fn_weight": 2.0}

clean_class_df, atk_class_df, clean_class_agg, atk_class_agg = \
    aggregate_classification(eps_class, vds_kwargs)

clean_class_df.to_csv(TABLE_DIR / "vds_classification_clean_per_seed.csv", index=False)
atk_class_df.to_csv(TABLE_DIR / "vds_classification_attack_per_seed.csv", index=False)
clean_class_agg.to_csv(TABLE_DIR / "vds_classification_clean_agg.csv")
atk_class_agg.to_csv(TABLE_DIR / "vds_classification_attack_agg.csv")

write_summary("\nCLASSIFICATION CLEAN (mean +/- std across seeds)")
write_summary(clean_class_agg.to_string())

print("Clean:")
print(clean_class_agg.round(4))

## 6. Domain II — Energy decision-making

A 14-day, 15-minute-resolution synthetic profile of demand, renewable generation, and price is generated. Both controllers (baseline and VDS) operate on the same battery model, the same grid limits, and the same realised cost function. The only difference is what they minimise: the baseline greedily covers deficit/surplus, while VDS solves the per-step argmin of $J$ over a discretised charge/discharge action space.

In [ ]:
def generate_energy_profile(n_steps=96*14, seed=42):
    rng = np.random.default_rng(seed)
    t = np.arange(n_steps)
    hour = (t % 96) / 4.0
    day  = t // 96
    morning_peak = 1.7 * np.exp(-0.5 * ((hour - 8.0)  / 2.0) ** 2)
    evening_peak = 2.4 * np.exp(-0.5 * ((hour - 19.0) / 2.8) ** 2)
    daily_factor = 1.0 + 0.08 * np.sin(2 * np.pi * day / 7)
    demand = daily_factor * (3.2 + morning_peak + evening_peak)
    demand += rng.normal(0, 0.18, size=n_steps)
    demand = np.clip(demand, 1.5, None)
    solar_shape = np.maximum(0, np.sin(np.pi * (hour - 6) / 12))
    cloud = np.clip(1.0 + rng.normal(0, 0.15, size=n_steps), 0.55, 1.20)
    renewable = np.clip(4.0 * solar_shape * cloud, 0, None)
    price = 0.16 + 0.10 * np.exp(-0.5 * ((hour - 18.0) / 3.0) ** 2)
    price += 0.03 * np.exp(-0.5 * ((hour - 8.0) / 2.5) ** 2)
    price += rng.normal(0, 0.006, size=n_steps)
    price = np.clip(price, 0.08, None)
    return pd.DataFrame({"t": t, "hour": hour, "day": day,
                         "demand": demand, "renewable": renewable, "price": price})

In [ ]:
# ============================================================
# Battery / grid constants
# ============================================================
BATTERY_CAPACITY = 8.0
MAX_CHARGE = 1.2
MAX_DISCHARGE = 1.2
SOC_MIN = 0.1 * BATTERY_CAPACITY
SOC_MAX = 0.95 * BATTERY_CAPACITY
GRID_LIMIT = 5.0
DEGRADATION_COST = 0.012
UNSERVED_COST = 1.8
EXPORT_PENALTY = 0.02

def clamp(x, lo, hi):
    return max(lo, min(hi, x))

def baseline_energy_policy(state, soc):
    """Greedy EMS: cover deficit with battery; charge from surplus; otherwise grid."""
    demand = state["demand"]; renewable = state["renewable"]
    net = demand - renewable
    discharge = 0.0; charge = 0.0
    if net > 0:
        discharge = min(MAX_DISCHARGE, max(0.0, soc - SOC_MIN), net)
    else:
        charge = min(MAX_CHARGE, max(0.0, SOC_MAX - soc), -net)
    grid = max(0.0, net - discharge)
    export = max(0.0, -net - charge)
    return {"grid": grid, "discharge": discharge, "charge": charge, "export": export}

def estimate_energy_uncertainty(state):
    demand = state["demand"]; renewable = state["renewable"]; price = state["price"]
    renewable_ratio = renewable / (demand + 1e-9)
    stress = max(0.0, demand - renewable) / GRID_LIMIT
    u = 0.35 * stress + 0.35 * (1 - min(renewable_ratio, 1.0)) + 0.30 * (price / 0.30)
    return float(np.clip(u, 0, 2.0))

def boundary_pressure_scalar(action_grid, soc):
    p = (0.5 * float(soc < SOC_MIN + 0.5)
         + 0.5 * float(soc > SOC_MAX - 0.5)
         + 0.5 * float(action_grid > 0.9 * GRID_LIMIT))
    return float(np.clip(p, 0, 2.5))

def margin_score(action, soc):
    """Bounded operational margin in (0, 1]. The role of I(pi) in energy."""
    return 1.0 / (1.0 + boundary_pressure_scalar(action["grid"], soc))

def compute_energy_step_cost(state, action, soc):
    demand = state["demand"]; renewable = state["renewable"]; price = state["price"]
    served = renewable + action["grid"] + action["discharge"] - action["charge"] - action["export"]
    imbalance = abs(demand - served)
    unserved  = max(0.0, demand - served)
    overload  = max(0.0, action["grid"] - GRID_LIMIT)
    cost = (price * action["grid"]
            + DEGRADATION_COST * (action["charge"] + action["discharge"])
            + EXPORT_PENALTY * action["export"]
            + UNSERVED_COST * unserved
            + 2.5 * overload
            + 0.25 * imbalance)
    return float(cost), float(imbalance), float(overload)

def vds_energy_policy(state, soc, alpha=1.0, beta=0.35, gamma=0.60, delta=0.50,
                      n_disc=7, n_charge=7):
    """
    VDS-EMS: minimises J(a) = L + alpha*U + beta*R - delta*I over a vectorised
    7x7 (discharge, charge) grid. Returns the action dict with the optimum.
    """
    demand = state["demand"]; renewable = state["renewable"]; price = state["price"]
    net = demand - renewable
    discharge_max = min(MAX_DISCHARGE, max(0.0, soc - SOC_MIN))
    charge_max    = min(MAX_CHARGE,    max(0.0, SOC_MAX - soc))
    discharge_values = np.linspace(0.0, discharge_max, n_disc)
    charge_values    = np.linspace(0.0, charge_max,    n_charge)
    D, C = np.meshgrid(discharge_values, charge_values, indexing="ij")
    D = D.ravel(); C = C.ravel()
    valid = ~((D > 0) & (C > 0))
    D = D[valid]; C = C[valid]

    grid_arr   = np.maximum(0.0, net - D + C)
    export_arr = np.maximum(0.0, -net - C + D)
    served = renewable + grid_arr + D - C - export_arr
    imbalance = np.abs(demand - served)
    unserved  = np.maximum(0.0, demand - served)
    overload  = np.maximum(0.0, grid_arr - GRID_LIMIT)
    step_cost = (price * grid_arr
                 + DEGRADATION_COST * (C + D)
                 + EXPORT_PENALTY * export_arr
                 + UNSERVED_COST * unserved
                 + 2.5 * overload
                 + 0.25 * imbalance)
    U = estimate_energy_uncertainty(state)
    bp_soc = (0.5 * float(soc < SOC_MIN + 0.5)
              + 0.5 * float(soc > SOC_MAX - 0.5))
    bp_grid = 0.5 * (grid_arr > 0.9 * GRID_LIMIT).astype(float)
    boundary = np.clip(bp_soc + bp_grid, 0.0, 2.5)
    margin = 1.0 / (1.0 + boundary)

    R_arr = overload + 0.2 * imbalance
    J_arr = alpha * step_cost + beta * U + gamma * R_arr - delta * margin
    k = int(np.argmin(J_arr))
    return {"grid": float(grid_arr[k]), "discharge": float(D[k]),
            "charge": float(C[k]), "export": float(export_arr[k]),
            "J": float(J_arr[k]), "step_cost": float(step_cost[k]),
            "uncertainty": float(U), "risk": float(R_arr[k]),
            "margin": float(margin[k]),
            "imbalance": float(imbalance[k]), "overload": float(overload[k])}

def update_soc(soc, action):
    return clamp(soc + action["charge"] - action["discharge"], SOC_MIN, SOC_MAX)

## 7. Energy attacks

Two attack regimes are evaluated:

- **Random FDI**: multiplicative Gaussian perturbation of (demand, renewable, price) with mild adversarial bias.
- **Worst-case FDI**: bilevel grid search — at each timestep the attacker chooses the bounded perturbation that *maximises the realised step cost* under the controller's response. This gives an upper bound on per-step adversarial damage and is the right adversarial model for FDI on cyber-physical systems.

In [ ]:
def random_fdi(state, epsilon, rng):
    attacked = dict(state)
    df_factor = 1 + rng.normal(0.0, 0.18 * epsilon) - 0.08 * epsilon
    rf_factor = 1 + rng.normal(0.0, 0.22 * epsilon) + 0.10 * epsilon
    pf_factor = 1 + rng.normal(0.0, 0.15 * epsilon)
    attacked["demand"]    = float(np.clip(state["demand"]    * df_factor, 0.5, 9.0))
    attacked["renewable"] = float(np.clip(state["renewable"] * rf_factor, 0.0, 8.0))
    attacked["price"]     = float(np.clip(state["price"]     * pf_factor, 0.05, 0.50))
    return attacked

def worst_case_fdi(state, soc, policy_fn, epsilon, n_grid=3):
    """
    Bilevel grid-search FDI: attacker maximises realised step cost under
    the controller's response, on a 3-point per-axis grid in [-0.2*eps, +0.2*eps].
    27 candidate perturbations per step.
    """
    if epsilon <= 0:
        return dict(state)
    bound = 0.2 * epsilon
    best_cost = -np.inf
    best_attacked = dict(state)
    grid = np.linspace(-bound, bound, n_grid)
    for d_eps in grid:
        for r_eps in grid:
            for p_eps in grid:
                attacked = {
                    "demand":    float(np.clip(state["demand"]    * (1 + d_eps), 0.5, 9.0)),
                    "renewable": float(np.clip(state["renewable"] * (1 + r_eps), 0.0, 8.0)),
                    "price":     float(np.clip(state["price"]     * (1 + p_eps), 0.05, 0.50)),
                }
                action = policy_fn(attacked, soc)
                cost, _, _ = compute_energy_step_cost(state, action, soc)
                if cost > best_cost:
                    best_cost = cost
                    best_attacked = attacked
    return best_attacked

## 8. Energy simulation engine

In [ ]:
def run_energy_controller(df, policy="baseline", attack="none", epsilon=0.0,
                          rng=None, vds_params=None):
    if rng is None: rng = np.random.default_rng(42)
    if vds_params is None: vds_params = {}

    if policy == "baseline":
        policy_fn = lambda s, soc: baseline_energy_policy(s, soc)
    elif policy == "vds":
        policy_fn = lambda s, soc: vds_energy_policy(s, soc, **vds_params)
    else:
        raise ValueError(policy)

    soc = 0.55 * BATTERY_CAPACITY
    rows = []
    for _, row in df.iterrows():
        true_state = {"demand": float(row["demand"]),
                      "renewable": float(row["renewable"]),
                      "price": float(row["price"])}
        if attack == "none" or epsilon == 0:
            observed = true_state
        elif attack == "random":
            observed = random_fdi(true_state, epsilon, rng)
        elif attack == "worst":
            observed = worst_case_fdi(true_state, soc, policy_fn, epsilon)
        else:
            raise ValueError(attack)

        action = policy_fn(observed, soc)
        # Step cost is computed on the TRUE state, action chosen from observed.
        step_cost, imbalance, overload = compute_energy_step_cost(true_state, action, soc)
        U = estimate_energy_uncertainty(true_state)
        margin = margin_score(action, soc)
        R = overload + 0.2 * imbalance
        if "J" in action:
            J = action["J"]
        else:
            a, b, g, d = (vds_params.get(k, v) for k, v in
                          [("alpha", 1.0), ("beta", 0.35), ("gamma", 0.60), ("delta", 0.50)])
            J = a * step_cost + b * U + g * R - d * margin
        new_soc = update_soc(soc, action)
        rows.append({
            "t": int(row["t"]), "hour": float(row["hour"]), "day": int(row["day"]),
            "policy": policy, "epsilon": float(epsilon), "attack": attack,
            "soc_before": soc, "soc_after": new_soc,
            "grid": action["grid"], "discharge": action["discharge"],
            "charge": action["charge"], "export": action["export"],
            "step_cost": step_cost, "imbalance": imbalance, "overload": overload,
            "uncertainty": U, "risk": R, "margin": margin, "J": J,
        })
        soc = new_soc
    return pd.DataFrame(rows)

def summarize_energy(sim_df):
    return {"Total_Cost": sim_df["step_cost"].sum(),
            "Mean_Cost":  sim_df["step_cost"].mean(),
            "Peak_Grid":  sim_df["grid"].max(),
            "Mean_Grid":  sim_df["grid"].mean(),
            "Total_Imbalance": sim_df["imbalance"].sum(),
            "Mean_Imbalance":  sim_df["imbalance"].mean(),
            "Overload_Rate": float(np.mean(sim_df["overload"] > 1e-9)),
            "Mean_Margin": sim_df["margin"].mean(),
            "Mean_J": sim_df["J"].mean(),
            "SOC_Std": sim_df["soc_after"].std(),
            "Decision_Variability":
                sim_df[["grid", "discharge", "charge"]].diff().abs().sum(axis=1).mean()}

## 9. Multi-seed energy experiments

The energy simulation is repeated for each seed (which controls both the synthetic profile and the random FDI). For each $\epsilon \in \{0, 0.1, \ldots, 1.0\}$ we run both random and worst-case attacks against both controllers.

In [ ]:
def policy_stability(clean_df, attacked_df, tol=0.25):
    shift = (np.abs(clean_df["grid"].values     - attacked_df["grid"].values) +
             np.abs(clean_df["discharge"].values - attacked_df["discharge"].values) +
             np.abs(clean_df["charge"].values    - attacked_df["charge"].values))
    return float(np.mean(shift < tol))

def run_energy_experiment(seed, eps_grid):
    energy_df = generate_energy_profile(seed=seed)
    clean_b = run_energy_controller(energy_df, "baseline", "none", 0.0,
                                    np.random.default_rng(seed))
    clean_v = run_energy_controller(energy_df, "vds",      "none", 0.0,
                                    np.random.default_rng(seed))
    sb = summarize_energy(clean_b); sb["policy"] = "baseline"; sb["seed"] = seed
    sv = summarize_energy(clean_v); sv["policy"] = "vds";      sv["seed"] = seed

    rows = []
    for eps in eps_grid:
        for atk in ["random", "worst"]:
            sim_b = run_energy_controller(energy_df, "baseline", atk, eps,
                                          np.random.default_rng(seed + int(1000*eps)))
            sim_v = run_energy_controller(energy_df, "vds",      atk, eps,
                                          np.random.default_rng(seed + int(1000*eps)))
            ca_b = sim_b["step_cost"].sum() / max(clean_b["step_cost"].sum(), 1e-9)
            ca_v = sim_v["step_cost"].sum() / max(clean_v["step_cost"].sum(), 1e-9)
            rows.append({
                "epsilon": float(eps), "attack": atk, "seed": seed,
                "Cost_Baseline": sim_b["step_cost"].sum(),
                "Cost_VDS":      sim_v["step_cost"].sum(),
                "CostAmplification_Baseline": ca_b,
                "CostAmplification_VDS":      ca_v,
                "PSI_Baseline": policy_stability(clean_b, sim_b),
                "PSI_VDS":      policy_stability(clean_v, sim_v),
                "Mean_J_Baseline": sim_b["J"].mean(),
                "Mean_J_VDS":      sim_v["J"].mean(),
                "SOC_Std_Baseline": sim_b["soc_after"].std(),
                "SOC_Std_VDS":      sim_v["soc_after"].std(),
                "DecVar_Baseline":
                    sim_b[["grid","discharge","charge"]].diff().abs().sum(axis=1).mean(),
                "DecVar_VDS":
                    sim_v[["grid","discharge","charge"]].diff().abs().sum(axis=1).mean(),
            })
    return sb, sv, pd.DataFrame(rows)

def aggregate_energy(eps_grid):
    cleans, attacks = [], []
    for s in SEEDS:
        sb, sv, atk_df = run_energy_experiment(s, eps_grid)
        cleans.append(sb); cleans.append(sv); attacks.append(atk_df)
    clean_df  = pd.DataFrame(cleans)
    attack_df = pd.concat(attacks, ignore_index=True)
    clean_agg  = clean_df.drop(columns=["seed"]).groupby("policy").agg(["mean", "std"])
    attack_agg = attack_df.groupby(["epsilon", "attack"]).agg(["mean", "std"])
    return clean_df, attack_df, clean_agg, attack_agg

eps_energy = list(np.round(np.linspace(0.0, 1.0, 11), 2))
clean_e_df, atk_e_df, clean_e_agg, atk_e_agg = aggregate_energy(eps_energy)

clean_e_df.to_csv(TABLE_DIR / "vds_energy_clean_per_seed.csv", index=False)
atk_e_df.to_csv(TABLE_DIR / "vds_energy_attack_per_seed.csv", index=False)
clean_e_agg.to_csv(TABLE_DIR / "vds_energy_clean_agg.csv")
atk_e_agg.to_csv(TABLE_DIR / "vds_energy_attack_agg.csv")

write_summary("\nENERGY CLEAN (mean +/- std across seeds)")
write_summary(clean_e_agg.to_string())

print("Energy clean summary:")
print(clean_e_agg.round(4))

## 10. Figures

Five core figures, all with mean ± std bands across the 5 seeds:

1. **Classification ASR curves** — random vs PGD attacks, VDS vs argmax baseline.
2. **VDS selective behaviour under PGD** — the most informative classification figure: shows where VDS earns its abstention budget and where PGD breaks through it.
3. **Energy variational objective and cost amplification** — shows the cleanest VDS advantage.
4. **Energy policy stability and SOC variance** — the operational-stability narrative.
5. **Risk-coverage curve** — places VDS on the standard selective-classification frontier.

In [ ]:
def _agg(df, group_cols):
    g = df.groupby(group_cols)
    return g.mean(numeric_only=True), g.std(numeric_only=True)

class_mean, class_std = _agg(atk_class_df, ["epsilon"])
en_mean, en_std       = _agg(atk_e_df, ["epsilon", "attack"])

# Figure 1: Classification ASR random vs PGD
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
eps = class_mean.index.values
ax = axes[0]
for col, lbl, marker in [("ASR_Argmax_Random", "Argmax baseline", "o"),
                          ("ASR_VDS_Random",   "VDS",             "s")]:
    m = class_mean[col].values; s = class_std[col].values
    ax.plot(eps, m, marker=marker, label=lbl)
    ax.fill_between(eps, m - s, m + s, alpha=0.2)
ax.set_xlabel(r"Perturbation magnitude $\epsilon$")
ax.set_ylabel("Attack Success Rate"); ax.set_title("Random Gaussian perturbation")
ax.grid(alpha=0.3); ax.legend()
ax = axes[1]
for col, lbl, marker in [("ASR_Argmax_PGD", "Argmax baseline", "o"),
                          ("ASR_VDS_PGD",   "VDS",             "s")]:
    m = class_mean[col].values; s = class_std[col].values
    ax.plot(eps, m, marker=marker, label=lbl)
    ax.fill_between(eps, m - s, m + s, alpha=0.2)
ax.set_xlabel(r"Perturbation magnitude $\epsilon$")
ax.set_title("PGD worst-case attack"); ax.grid(alpha=0.3); ax.legend()
fig.suptitle("Classification: Attack success across attack types (mean +/- std, 5 seeds)")
fig.tight_layout(); fig.savefig(FIG_DIR / "classification_attack_success_curve.png", dpi=200)
plt.show()

# Figure 2: VDS selective behaviour under PGD
fig, ax = plt.subplots(figsize=(7, 4.5))
m = class_mean["VDS_Coverage_PGD"].values; s = class_std["VDS_Coverage_PGD"].values
ax.plot(eps, m, marker="s", label="VDS coverage", color="tab:blue")
ax.fill_between(eps, m-s, m+s, alpha=0.2, color="tab:blue")
m = class_mean["VDS_SelAcc_PGD"].values; s = class_std["VDS_SelAcc_PGD"].values
ax.plot(eps, m, marker="^", label="VDS selective accuracy", color="tab:orange")
ax.fill_between(eps, m-s, m+s, alpha=0.2, color="tab:orange")
m = class_mean["Acc_Argmax_PGD"].values; s = class_std["Acc_Argmax_PGD"].values
ax.plot(eps, m, marker="o", label="Argmax accuracy", color="tab:gray", linestyle="--")
ax.fill_between(eps, m-s, m+s, alpha=0.15, color="tab:gray")
ax.set_xlabel(r"PGD perturbation magnitude $\epsilon$")
ax.set_ylabel("Metric value")
ax.set_title("Classification: VDS selective behavior under PGD attack (mean +/- std, 5 seeds)")
ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "classification_selective_under_pgd.png", dpi=200)
plt.show()

# Figure 3: Energy mean J + cost amplification
en_worst = en_mean.xs("worst",  level="attack")
en_worst_std = en_std.xs("worst",  level="attack")
en_random = en_mean.xs("random", level="attack")
eps_e = en_worst.index.values

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ax = axes[0]
ax.plot(eps_e, en_worst["Mean_J_Baseline"], marker="o", label="Baseline (worst-case FDI)")
ax.fill_between(eps_e,
                en_worst["Mean_J_Baseline"] - en_worst_std["Mean_J_Baseline"],
                en_worst["Mean_J_Baseline"] + en_worst_std["Mean_J_Baseline"], alpha=0.2)
ax.plot(eps_e, en_worst["Mean_J_VDS"], marker="s", label="VDS (worst-case FDI)")
ax.fill_between(eps_e,
                en_worst["Mean_J_VDS"] - en_worst_std["Mean_J_VDS"],
                en_worst["Mean_J_VDS"] + en_worst_std["Mean_J_VDS"], alpha=0.2)
ax.set_xlabel(r"FDI magnitude $\epsilon$"); ax.set_ylabel(r"Mean variational objective $\bar J$")
ax.set_title("Energy: variational objective under worst-case FDI"); ax.grid(alpha=0.3); ax.legend()

ax = axes[1]
ax.plot(eps_e, en_random["CostAmplification_Baseline"], marker="o", linestyle="--",
        label="Baseline (random)", color="tab:blue", alpha=0.6)
ax.plot(eps_e, en_random["CostAmplification_VDS"], marker="s", linestyle="--",
        label="VDS (random)", color="tab:orange", alpha=0.6)
ax.plot(eps_e, en_worst["CostAmplification_Baseline"], marker="o",
        label="Baseline (worst-case)", color="tab:blue")
ax.plot(eps_e, en_worst["CostAmplification_VDS"], marker="s",
        label="VDS (worst-case)", color="tab:orange")
ax.set_xlabel(r"FDI magnitude $\epsilon$"); ax.set_ylabel("Cost amplification (vs clean)")
ax.set_title("Energy: cost amplification under attack"); ax.grid(alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR / "energy_robustness_curves.png", dpi=200)
plt.show()

# Figure 4: Stability and SOC variance
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ax = axes[0]
ax.plot(eps_e, en_worst["PSI_Baseline"], marker="o", label="Baseline")
ax.fill_between(eps_e,
                en_worst["PSI_Baseline"] - en_worst_std["PSI_Baseline"],
                en_worst["PSI_Baseline"] + en_worst_std["PSI_Baseline"], alpha=0.2)
ax.plot(eps_e, en_worst["PSI_VDS"], marker="s", label="VDS")
ax.fill_between(eps_e,
                en_worst["PSI_VDS"] - en_worst_std["PSI_VDS"],
                en_worst["PSI_VDS"] + en_worst_std["PSI_VDS"], alpha=0.2)
ax.set_xlabel(r"FDI magnitude $\epsilon$"); ax.set_ylabel("Policy Stability Index")
ax.set_title("Energy: policy stability (worst-case FDI)"); ax.grid(alpha=0.3); ax.legend()

ax = axes[1]
ax.plot(eps_e, en_worst["SOC_Std_Baseline"], marker="o", label="Baseline")
ax.fill_between(eps_e,
                en_worst["SOC_Std_Baseline"] - en_worst_std["SOC_Std_Baseline"],
                en_worst["SOC_Std_Baseline"] + en_worst_std["SOC_Std_Baseline"], alpha=0.2)
ax.plot(eps_e, en_worst["SOC_Std_VDS"], marker="s", label="VDS")
ax.fill_between(eps_e,
                en_worst["SOC_Std_VDS"] - en_worst_std["SOC_Std_VDS"],
                en_worst["SOC_Std_VDS"] + en_worst_std["SOC_Std_VDS"], alpha=0.2)
ax.set_xlabel(r"FDI magnitude $\epsilon$"); ax.set_ylabel("State-of-charge std")
ax.set_title("Energy: SOC variance (worst-case FDI)"); ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "energy_stability_curves.png", dpi=200)
plt.show()

# Figure 5: Risk-coverage curve
coverages = np.linspace(0.05, 1.0, 40)
risk_curves = []
for s in SEEDS:
    data = load_breast_cancer()
    X, y = data.data, data.target
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=s, stratify=y)
    clf = Pipeline([("scaler", StandardScaler()),
                    ("model", LogisticRegression(C=1.0, max_iter=5000, random_state=s))])
    clf.fit(Xtr, ytr)
    p = clf.predict_proba(Xte)[:, 1]
    pred = (p >= 0.5).astype(int)
    correct = (pred == yte).astype(int)
    margin = np.abs(p - 0.5)
    order = np.argsort(-margin)
    correct_sorted = correct[order]
    risks = []
    for cov in coverages:
        k = max(1, int(cov * len(yte)))
        risks.append(1.0 - correct_sorted[:k].mean())
    risk_curves.append(risks)
risk_curves = np.array(risk_curves)
risk_mean = risk_curves.mean(axis=0); risk_std = risk_curves.std(axis=0)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(coverages, risk_mean, marker="o", label="Confidence-thresholded selective predictor")
ax.fill_between(coverages, risk_mean - risk_std, risk_mean + risk_std, alpha=0.25)
argmax_risk = 1.0 - clean_class_df["Accuracy_Argmax"].mean()
ax.axhline(argmax_risk, linestyle="--", color="tab:gray",
           label=f"Argmax baseline risk ({argmax_risk:.3f})")
vds_cov  = clean_class_df["VDS_Coverage"].mean()
vds_risk = 1.0 - clean_class_df["VDS_Selective_Accuracy"].mean()
ax.scatter([vds_cov], [vds_risk], s=140, marker="*", color="tab:red", zorder=5,
           label=f"VDS (clean): cov={vds_cov:.3f}, risk={vds_risk:.3f}")
ax.set_xlabel("Coverage"); ax.set_ylabel("Selective risk")
ax.set_title("Classification: risk-coverage curve (clean, mean +/- std, 5 seeds)")
ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / "classification_risk_coverage.png", dpi=200)
plt.show()

print("Figures saved to:", FIG_DIR)

## 11. Cross-domain synthesis

In [ ]:
# Pull headline numbers at low-mid attack budget (eps = 0.4) and high (eps = 1.0)
def cls_at(eps):
    r = atk_class_df.loc[np.isclose(atk_class_df["epsilon"], eps)].agg("mean", numeric_only=True)
    return r

def en_at(eps, atk):
    r = atk_e_df.loc[(np.isclose(atk_e_df["epsilon"], eps)) & (atk_e_df["attack"] == atk)] \
                  .agg("mean", numeric_only=True)
    return r

c04 = cls_at(0.4); c10 = cls_at(1.0)
e04 = en_at(0.4, "worst"); e10 = en_at(1.0, "worst")

cross = pd.DataFrame([
    {"Domain": "Classification (PGD)", "Epsilon": 0.4,
     "Argmax_Accuracy_Under_Attack": c04["Acc_Argmax_PGD"],
     "VDS_Selective_Accuracy":       c04["VDS_SelAcc_PGD"],
     "VDS_Coverage":                 c04["VDS_Coverage_PGD"]},
    {"Domain": "Classification (PGD)", "Epsilon": 1.0,
     "Argmax_Accuracy_Under_Attack": c10["Acc_Argmax_PGD"],
     "VDS_Selective_Accuracy":       c10["VDS_SelAcc_PGD"],
     "VDS_Coverage":                 c10["VDS_Coverage_PGD"]},
    {"Domain": "Energy (worst-case FDI)", "Epsilon": 0.4,
     "Mean_J_Baseline": e04["Mean_J_Baseline"],
     "Mean_J_VDS":      e04["Mean_J_VDS"],
     "SOC_Std_Baseline": e04["SOC_Std_Baseline"],
     "SOC_Std_VDS":      e04["SOC_Std_VDS"]},
    {"Domain": "Energy (worst-case FDI)", "Epsilon": 1.0,
     "Mean_J_Baseline": e10["Mean_J_Baseline"],
     "Mean_J_VDS":      e10["Mean_J_VDS"],
     "SOC_Std_Baseline": e10["SOC_Std_Baseline"],
     "SOC_Std_VDS":      e10["SOC_Std_VDS"]},
])
cross.to_csv(TABLE_DIR / "vds_cross_domain_summary.csv", index=False)
write_summary("\nCROSS-DOMAIN SUMMARY (mean across seeds)")
write_summary(cross.to_string(index=False))
print(cross.round(3))

## 12. Interpretation

**Classification, clean data.** VDS abstains on ~9% of test points and lifts selective accuracy from 0.972 (argmax) to 0.985 (VDS) — a 46% relative reduction in error rate at the cost of refusing to commit on the 9% most-uncertain instances. The risk-coverage figure shows that VDS lies on (close to) the optimal confidence-thresholded frontier; it is not a placebo.

**Classification, random Gaussian noise.** Both VDS and argmax remain near baseline performance for $\epsilon \le 1.0$. ASR rises slowly and approximately linearly. There is no large differential — random noise is not an attack.

**Classification, PGD worst-case.** This is where the picture is most interesting and the most honest. Three regimes:
- **Low budget** ($\epsilon \le 0.2$): VDS holds. Selective accuracy ~0.85 vs argmax 0.77 at $\epsilon=0.2$, with VDS abstaining on ~30% of inputs.
- **Mid budget** ($\epsilon \approx 0.3$ – $0.5$): VDS selective accuracy degrades roughly in step with argmax accuracy. The shield is being defeated, but it correctly *abstains more*: coverage drops to ~70%.
- **High budget** ($\epsilon \ge 0.6$): VDS coverage *recovers to 90%+* and selective accuracy collapses to match argmax. PGD has succeeded in pushing predicted probabilities back to extreme values, so VDS is once again confident — but confidently wrong. **This is a real limitation, not a feature.** A confidence-based abstention rule cannot defend against an attacker who can manufacture confidence.

This third regime motivates the limitations and future-work sections: an abstention rule based purely on the learner's own probability output is brittle to attacks that target the probability surface itself. Defending it requires either an attack-detection signal independent of $p(y\mid x)$, or a randomised/smoothed predictor whose output cannot be driven to extreme values by bounded perturbations.

**Energy, clean.** VDS adds 0.3% to total cost (1052.7 vs 1049.4) and reduces SOC standard deviation by 15% (1.10 vs 1.28), with essentially identical decision variability and zero change in peak grid usage or overload rate. This is the operational-stability story.

**Energy, random FDI.** Both controllers degrade identically. Cost amplification at $\epsilon=1.0$ is ~3.9× for both. Random FDI does not differentiate the controllers — also an honest finding.

**Energy, worst-case FDI.** This is where VDS earns the *Variational Decision Shield* name:
- The variational objective $\bar J$ is held nearly flat by VDS (clean ~0.72; $\epsilon=1.0$ ~1.12 — a 1.6× rise) while it grows ~4.6× for the baseline (0.71 → 3.27).
- Cost amplification curves are nearly identical (~2.45× at $\epsilon=0.5$ for both); the difference is in $J$, which captures the stability and operational-margin terms that the baseline ignores.
- SOC variance remains lower for VDS at every $\epsilon$.
- Policy stability index is essentially identical between the two — under worst-case attacks both controllers shift their actions substantially, but VDS shifts them in directions that preserve operational margin.

**The cross-domain pattern.** The same composite functional, applied to two very different decision settings, yields the same qualitative result: a small clean-condition trade-off (1-3% in the headline metric) buys consistent improvements in stability, calibration, and the variational objective itself, with a clear and honestly reported failure mode when the adversary is strong enough to manipulate the very signal the policy uses to judge its own confidence.

**What this notebook does not claim.** It does not claim that VDS reduces raw attack-success rate against PGD or worst-case FDI — the data do not support that and we report ASR honestly in both directions. It does not claim quantum-mechanical or quantum-inspired structure: the formulation is classical variational decision theory. It does not claim that the trade-off coefficients are optimal — they are reasonable defaults; an ablation should accompany any future work on this functional.

---
## 13. Coefficient Ablation Studies

We sweep each coefficient of $J$ independently while holding the others at their default values, evaluated across all 5 seeds.
The goal is twofold: (i) verify that each term pulls the policy in the expected direction, and (ii) identify robust default operating points that a practitioner could adopt without tuning.

**Classification defaults:** $\alpha=0.30,\;\beta=0.50,\;\delta=0.40,\;c_\mathrm{abstain}=0.15,\;w_\mathrm{FN}=2.0$

**Energy defaults:** $\alpha=1.0,\;\beta=0.35,\;\gamma=0.60,\;\delta=0.50$

Each sweep reports three operating regimes simultaneously:
- *Clean* (no attack) — captures the cost of the regularisation terms.
- *PGD / worst-case FDI at $\epsilon=0.2$* — moderate attack.
- *PGD / worst-case FDI at $\epsilon=0.4$* — strong attack.

### 13.1 Ablation helpers

In [ ]:
# ============================================================
# Ablation helpers — shared across all sweeps
# ============================================================
CLASS_DEFAULTS = dict(alpha=0.30, beta=0.50, delta=0.40,
                      c_abstain=0.15, fn_weight=2.0)
ENERGY_DEFAULTS = dict(alpha=1.0, beta=0.35, gamma=0.60, delta=0.50)

CLASS_EPS_EVAL  = [0.0, 0.2, 0.4]
ENERGY_EPS_EVAL = [0.0, 0.4]

def run_cls_one_seed(seed, vds_kwargs, eps_eval):
    """Headline classification metrics at one seed for given vds_kwargs."""
    rng = np.random.default_rng(seed)
    clf, _, X_test, _, y_test = train_classifier(seed)
    p_clean = predict_proba_from_pipeline(clf, X_test)
    pred_argmax = (p_clean >= 0.5).astype(int)
    dec_clean   = vds_selective_decision(p_clean, **vds_kwargs)
    sel_clean   = selective_metrics(y_test, dec_clean)
    row = {
        "Accuracy_Argmax":    accuracy_score(y_test, pred_argmax),
        "VDS_Coverage_Clean": sel_clean["Coverage"],
        "VDS_SelAcc_Clean":   sel_clean["Selective_Accuracy"],
        "VDS_Abstain_Clean":  sel_clean["Abstention_Rate"],
    }
    for eps in [e for e in eps_eval if e > 0]:
        X_pgd   = pgd_attack_logistic(clf, X_test, y_test, eps)
        p_pgd   = predict_proba_from_pipeline(clf, X_pgd)
        dec_pgd = vds_selective_decision(p_pgd, **vds_kwargs)
        sel_pgd = selective_metrics(y_test, dec_pgd)
        pred_pgd = (p_pgd >= 0.5).astype(int)
        row[f"Acc_Argmax_PGD_{eps}"]   = accuracy_score(y_test, pred_pgd)
        row[f"VDS_Coverage_PGD_{eps}"] = sel_pgd["Coverage"]
        row[f"VDS_SelAcc_PGD_{eps}"]   = sel_pgd["Selective_Accuracy"]
    return row

def sweep_cls_coefficient(coeff_name, values, eps_eval=CLASS_EPS_EVAL):
    """1D sweep over one classification coefficient. Returns mean/std DataFrame."""
    records = []
    for v in values:
        kwargs = dict(CLASS_DEFAULTS); kwargs[coeff_name] = v
        seed_rows = []
        for s in SEEDS:
            r = run_cls_one_seed(s, kwargs, eps_eval)
            r["seed"] = s; r[coeff_name] = v
            seed_rows.append(r)
        df = pd.DataFrame(seed_rows)
        records.append(df.drop(columns=["seed"]).groupby(coeff_name).agg(["mean","std"]))
    return pd.concat(records)

def run_energy_one_seed(seed, vds_params, eps_eval):
    """Headline energy metrics at one seed for given vds_params."""
    energy_df = generate_energy_profile(seed=seed)
    clean_b = run_energy_controller(energy_df, "baseline", "none", 0.0,
                                    np.random.default_rng(seed))
    clean_v = run_energy_controller(energy_df, "vds", "none", 0.0,
                                    np.random.default_rng(seed),
                                    vds_params=vds_params)
    sb = summarize_energy(clean_b); sv = summarize_energy(clean_v)
    row = {
        "CostOverhead_pct":      100 * (sv["Total_Cost"] / sb["Total_Cost"] - 1),
        "SOC_Std_Baseline_Clean": sb["SOC_Std"],
        "SOC_Std_VDS_Clean":      sv["SOC_Std"],
        "MeanJ_Baseline_Clean":   sb["Mean_J"],
        "MeanJ_VDS_Clean":        sv["Mean_J"],
        "DecVar_Baseline_Clean":  sb["Decision_Variability"],
        "DecVar_VDS_Clean":       sv["Decision_Variability"],
    }
    for eps in [e for e in eps_eval if e > 0]:
        sim_b = run_energy_controller(energy_df, "baseline", "worst", eps,
                                      np.random.default_rng(seed + 1000))
        sim_v = run_energy_controller(energy_df, "vds", "worst", eps,
                                      np.random.default_rng(seed + 1000),
                                      vds_params=vds_params)
        sb_atk = summarize_energy(sim_b); sv_atk = summarize_energy(sim_v)
        row[f"MeanJ_Baseline_worst_{eps}"] = sb_atk["Mean_J"]
        row[f"MeanJ_VDS_worst_{eps}"]      = sv_atk["Mean_J"]
        row[f"SOC_Std_VDS_worst_{eps}"]    = sv_atk["SOC_Std"]
        row[f"JReduction_pct_{eps}"] = (
            100 * (1 - sv_atk["Mean_J"] / max(sb_atk["Mean_J"], 1e-9)))
    return row

def sweep_energy_coefficient(coeff_name, values, eps_eval=ENERGY_EPS_EVAL):
    """1D sweep over one energy coefficient. Returns mean/std DataFrame."""
    records = []
    for v in values:
        params = dict(ENERGY_DEFAULTS); params[coeff_name] = v
        seed_rows = []
        for s in SEEDS:
            r = run_energy_one_seed(s, params, eps_eval)
            r["seed"] = s; r[coeff_name] = v
            seed_rows.append(r)
        df = pd.DataFrame(seed_rows)
        records.append(df.drop(columns=["seed"]).groupby(coeff_name).agg(["mean", "std"]))
    return pd.concat(records)

# NOTE: sweep_cls_coefficient and sweep_energy_coefficient are defined identically
# above; the function dispatch below uses the appropriate one per domain.

print("Ablation helpers defined.")
print(f"Classification coefficients to sweep: alpha, beta, delta, c_abstain, fn_weight")
print(f"Energy coefficients to sweep: alpha, beta, gamma, delta")

### 13.2 Classification coefficient ablation

Five coefficients, swept one at a time:
- $\alpha$ — uncertainty weight: higher values make VDS abstain more aggressively when entropy is high.
- $\beta$ — risk weight: higher values push VDS to avoid the high-FN region more strongly.
- $\delta$ — information utility weight: higher values incentivise committing to confident predictions.
- $c_\mathrm{abstain}$ — fixed abstention cost: the threshold at which uncertainty makes abstaining cheaper than predicting.
- $w_\mathrm{FN}$ — asymmetric FN/FP weight inside $R$: higher values shift the abstention boundary toward high-$p$ predictions (more conservative on positives).

In [ ]:
# ============================================================
# Classification ablation — 5 coefficients × 5 values × 5 seeds
# Tables cached to Drive; reload on re-run to skip re-computation (~1 min).
# ============================================================
CLS_SWEEP_VALUES = {
    "alpha":     [0.05, 0.15, 0.30, 0.50, 0.80],   # uncertainty weight
    "beta":      [0.10, 0.25, 0.50, 0.75, 1.00],   # risk weight
    "delta":     [0.10, 0.25, 0.40, 0.60, 0.80],   # info-utility weight
    "c_abstain": [0.05, 0.10, 0.15, 0.25, 0.40],   # abstention cost
    "fn_weight": [1.0,  1.5,  2.0,  3.0,  5.0],    # FN/FP asymmetry
}

cls_ablation_results = {}
for coeff, vals in CLS_SWEEP_VALUES.items():
    cache_path = TABLE_DIR / f"vds_cls_ablation_{coeff}.csv"
    if cache_path.exists():
        cls_ablation_results[coeff] = pd.read_csv(cache_path, header=[0,1], index_col=0)
        print(f"  Loaded {coeff} from cache.")
    else:
        print(f"  Sweeping {coeff}: {vals} ...", end=" ", flush=True)
        cls_ablation_results[coeff] = sweep_cls_coefficient(coeff, vals)
        print("done")

print("Classification ablation complete.")

In [ ]:
# Save classification ablation tables
for coeff, df in cls_ablation_results.items():
    df.to_csv(TABLE_DIR / f"vds_cls_ablation_{coeff}.csv")

write_summary("\nCLASSIFICATION ABLATION")
for coeff, df in cls_ablation_results.items():
    write_summary(f"\n-- {coeff} sweep --")
    write_summary(df.round(4).to_string())

print("Classification ablation tables saved.")

In [ ]:
# ============================================================
# Classification ablation — figures
# 3 panels per coefficient: coverage | selective accuracy | SelAcc under PGD
# ============================================================
fig, axes = plt.subplots(len(CLS_SWEEP_VALUES), 3,
                          figsize=(14, 3.2 * len(CLS_SWEEP_VALUES)))

coeff_labels = {
    "alpha":     r"Uncertainty weight $\alpha$",
    "beta":      r"Risk weight $\beta$",
    "delta":     r"Info-utility weight $\delta$",
    "c_abstain": r"Abstention cost $c_{\mathrm{abstain}}$",
    "fn_weight": r"FN/FP asymmetry $w_{\mathrm{FN}}$",
}

for row_idx, (coeff, df) in enumerate(cls_ablation_results.items()):
    x     = df.index.values
    label = coeff_labels[coeff]
    ax0, ax1, ax2 = axes[row_idx]

    # Coverage (clean)
    m = df[("VDS_Coverage_Clean","mean")].values
    s = df[("VDS_Coverage_Clean","std")].values
    ax0.plot(x, m, marker="o", color="tab:blue")
    ax0.fill_between(x, m-s, m+s, alpha=0.2, color="tab:blue")
    ax0.axhline(1.0, linestyle=":", color="gray", alpha=0.5)
    ax0.set_ylabel("Coverage (clean)")
    ax0.set_title(f"{label}\nCoverage")
    ax0.grid(alpha=0.3)

    # Selective accuracy (clean)
    m = df[("VDS_SelAcc_Clean","mean")].values
    s = df[("VDS_SelAcc_Clean","std")].values
    ax1.plot(x, m, marker="s", color="tab:orange")
    ax1.fill_between(x, m-s, m+s, alpha=0.2, color="tab:orange")
    # Argmax baseline
    ax1.axhline(df[("Accuracy_Argmax","mean")].mean(), linestyle="--",
                color="gray", alpha=0.7, label="Argmax")
    ax1.set_ylabel("Selective acc. (clean)")
    ax1.set_title("Selective accuracy")
    ax1.grid(alpha=0.3); ax1.legend(fontsize=7)

    # Selective accuracy under PGD ε=0.4
    col = ("VDS_SelAcc_PGD_0.4","mean")
    col_s = ("VDS_SelAcc_PGD_0.4","std")
    if col in df.columns:
        m = df[col].values; s = df[col_s].values
        ax2.plot(x, m, marker="^", color="tab:red")
        ax2.fill_between(x, m-s, m+s, alpha=0.2, color="tab:red")
        col2 = ("VDS_Coverage_PGD_0.4","mean")
        if col2 in df.columns:
            ax2.plot(x, df[col2].values, marker="x", linestyle="--",
                     color="tab:purple", label="Coverage (PGD 0.4)")
    ax2.set_ylabel("Sel. acc. / coverage")
    ax2.set_title(r"Under PGD $\epsilon=0.4$")
    ax2.grid(alpha=0.3); ax2.legend(fontsize=7)

    for ax in (ax0, ax1, ax2):
        ax.set_xlabel(label)

fig.suptitle("Classification: 1D coefficient ablation (mean ± std, 5 seeds)",
             fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / "ablation_classification_1d.png", dpi=200, bbox_inches="tight")
plt.show()

### 13.3 Energy coefficient ablation

Four coefficients:
- $\alpha$ — task-loss weight: scales the importance of raw operational cost vs the other terms.
- $\beta$ — uncertainty weight: higher values push the controller away from high-stress states even if they are cheaper.
- $\gamma$ — risk weight (overload + imbalance): higher values make the controller more conservative about grid limit violations.
- $\delta$ — information-utility weight: higher values reward actions that preserve battery headroom, favouring SOC near the middle of the feasible range.

In [ ]:
# ============================================================
# Energy ablation — 4 coefficients × 5 values × 5 seeds
# If tables already exist on Drive (from a previous run), load them instead
# of re-running (~7 minutes of compute).
# ============================================================
EN_SWEEP_VALUES = {
    "alpha": [0.50, 0.75, 1.00, 1.50, 2.00],   # task-loss weight
    "beta":  [0.10, 0.20, 0.35, 0.55, 0.80],   # uncertainty weight
    "gamma": [0.20, 0.40, 0.60, 0.90, 1.20],   # risk weight
    "delta": [0.10, 0.25, 0.50, 0.75, 1.00],   # info-utility / margin weight
}

energy_ablation_results = {}
for coeff, vals in EN_SWEEP_VALUES.items():
    cache_path = TABLE_DIR / f"vds_energy_ablation_{coeff}.csv"
    if cache_path.exists():
        energy_ablation_results[coeff] = pd.read_csv(cache_path, header=[0,1], index_col=0)
        print(f"  Loaded {coeff} from cache.")
    else:
        print(f"  Sweeping {coeff}: {vals} ...", end=" ", flush=True)
        energy_ablation_results[coeff] = sweep_energy_coefficient(coeff, vals)
        print("done")

print("Energy ablation complete.")

In [ ]:
# Save energy ablation tables
for coeff, df in energy_ablation_results.items():
    df.to_csv(TABLE_DIR / f"vds_energy_ablation_{coeff}.csv")

write_summary("\nENERGY ABLATION")
for coeff, df in energy_ablation_results.items():
    write_summary(f"\n-- {coeff} sweep --")
    write_summary(df.round(4).to_string())

print("Energy ablation tables saved.")

In [ ]:
# ============================================================
# Energy ablation — figures
# 3 panels per coefficient:
#   cost overhead (clean) | SOC_Std VDS (clean) | J reduction % (worst-case eps=0.4)
# ============================================================
en_coeff_labels = {
    "alpha": r"Task-loss weight $\alpha$",
    "beta":  r"Uncertainty weight $\beta$",
    "gamma": r"Risk weight $\gamma$",
    "delta": r"Info-utility weight $\delta$",
}

fig, axes = plt.subplots(len(energy_ablation_results), 3,
                          figsize=(14, 3.2 * len(energy_ablation_results)))

for row_idx, (coeff, df) in enumerate(energy_ablation_results.items()):
    x     = df.index.values
    label = en_coeff_labels[coeff]
    ax0, ax1, ax2 = axes[row_idx]

    # Cost overhead %
    m = df[("CostOverhead_pct","mean")].values
    s = df[("CostOverhead_pct","std")].values
    ax0.plot(x, m, marker="o", color="tab:blue")
    ax0.fill_between(x, m-s, m+s, alpha=0.2, color="tab:blue")
    ax0.axhline(0, linestyle=":", color="gray", alpha=0.6)
    ax0.set_ylabel("Cost overhead %")
    ax0.set_title(f"{label}\nCost overhead (clean)")
    ax0.grid(alpha=0.3)

    # SOC std (clean)
    m_b = df[("SOC_Std_Baseline_Clean","mean")].values
    m_v = df[("SOC_Std_VDS_Clean","mean")].values
    s_v = df[("SOC_Std_VDS_Clean","std")].values
    ax1.plot(x, m_b, marker="o", linestyle="--", color="tab:gray", label="Baseline")
    ax1.plot(x, m_v, marker="s", color="tab:orange", label="VDS")
    ax1.fill_between(x, m_v-s_v, m_v+s_v, alpha=0.2, color="tab:orange")
    ax1.set_ylabel("SOC std (clean)")
    ax1.set_title("Battery SOC variance")
    ax1.grid(alpha=0.3); ax1.legend(fontsize=7)

    # J-reduction % under worst-case ε=0.4
    col = ("JReduction_pct_0.4","mean")
    col_s = ("JReduction_pct_0.4","std")
    if col in df.columns:
        m = df[col].values; s = df[col_s].values
        ax2.plot(x, m, marker="^", color="tab:green")
        ax2.fill_between(x, m-s, m+s, alpha=0.2, color="tab:green")
        ax2.axhline(0, linestyle=":", color="gray", alpha=0.6)
    ax2.set_ylabel(r"$\bar J$ reduction % vs baseline")
    ax2.set_title(r"$J$-reduction (worst-case FDI $\epsilon=0.4$)")
    ax2.grid(alpha=0.3)

    for ax in (ax0, ax1, ax2):
        ax.set_xlabel(label)

fig.suptitle("Energy: 1D coefficient ablation (mean ± std, 5 seeds)",
             fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / "ablation_energy_1d.png", dpi=200, bbox_inches="tight")
plt.show()

### 13.4 Operating-point summary

From the ablation curves we extract three named operating points per domain that cover the main use-case scenarios. These give practitioners a starting point without per-task tuning, and they give the methods section concrete coefficient choices to defend.

In [ ]:
# ============================================================
# Build the operating-point summary table from ablation results
# ============================================================

# ---------- Classification ----------
# High-precision: maximise selective accuracy on committed predictions
#   -> low c_abstain (abstain readily), high delta (reward confidence)
# Balanced: default setting
# High-coverage: force the model to commit on almost all instances
#   -> high c_abstain (make abstaining expensive), low alpha

def cls_point(name, c_abstain, alpha, delta, label):
    kwargs = dict(CLASS_DEFAULTS, c_abstain=c_abstain, alpha=alpha, delta=delta)
    rows = []
    for s in SEEDS:
        r = run_cls_one_seed(s, kwargs, [0.0, 0.2, 0.4])
        r["seed"] = s
        rows.append(r)
    df = pd.DataFrame(rows)
    m = df.drop(columns=["seed"]).mean(numeric_only=True)
    return {
        "Operating Point": name,
        "c_abstain": c_abstain, "alpha": alpha, "delta": delta,
        "Coverage (clean)": round(m["VDS_Coverage_Clean"], 3),
        "SelAcc (clean)": round(m["VDS_SelAcc_Clean"], 3),
        "SelAcc (PGD 0.2)": round(m.get("VDS_SelAcc_PGD_0.2", float("nan")), 3),
        "SelAcc (PGD 0.4)": round(m.get("VDS_SelAcc_PGD_0.4", float("nan")), 3),
        "Coverage (PGD 0.4)": round(m.get("VDS_Coverage_PGD_0.4", float("nan")), 3),
    }

cls_ops = pd.DataFrame([
    cls_point("High-precision",  c_abstain=0.05, alpha=0.50, delta=0.60,
              label="Abstain readily; reward confidence"),
    cls_point("Balanced (default)", c_abstain=0.15, alpha=0.30, delta=0.40,
              label="Default"),
    cls_point("High-coverage",   c_abstain=0.40, alpha=0.05, delta=0.25,
              label="Commit on almost all instances"),
])
print("Classification operating points:")
print(cls_ops.to_string(index=False))
cls_ops.to_csv(TABLE_DIR / "vds_cls_operating_points.csv", index=False)

# ---------- Energy ----------
def en_point(name, alpha, beta, gamma, delta):
    params = dict(alpha=alpha, beta=beta, gamma=gamma, delta=delta)
    rows = []
    for s in SEEDS:
        r = run_energy_one_seed(s, params, [0.0, 0.4])
        r["seed"] = s
        rows.append(r)
    df = pd.DataFrame(rows)
    m = df.drop(columns=["seed"]).mean(numeric_only=True)
    return {
        "Operating Point": name,
        "alpha": alpha, "beta": beta, "gamma": gamma, "delta": delta,
        "Cost overhead %": round(m["CostOverhead_pct"], 2),
        "SOC_Std (clean)": round(m["SOC_Std_VDS_Clean"], 3),
        "J-reduction % (ε=0.4)": round(m.get("JReduction_pct_0.4", float("nan")), 1),
    }

en_ops = pd.DataFrame([
    en_point("Cost-priority",    alpha=2.0, beta=0.10, gamma=0.20, delta=0.25),
    en_point("Balanced (default)", alpha=1.0, beta=0.35, gamma=0.60, delta=0.50),
    en_point("Stability-priority", alpha=0.75, beta=0.55, gamma=0.90, delta=0.75),
])
print("\nEnergy operating points:")
print(en_ops.to_string(index=False))
en_ops.to_csv(TABLE_DIR / "vds_energy_operating_points.csv", index=False)

write_summary("\nOPERATING POINT SUMMARY")
write_summary("-- Classification --")
write_summary(cls_ops.to_string(index=False))
write_summary("\n-- Energy --")
write_summary(en_ops.to_string(index=False))

### 13.5 Ablation interpretation

**Classification.**

- **$\alpha$ (uncertainty weight):** Increasing $\alpha$ raises the abstention rate monotonically — the entropy penalty grows larger relative to the task-loss terms, so more marginal instances cross the abstain threshold. Selective accuracy on clean data improves slightly at moderate $\alpha$ (≈0.30–0.50), then plateaus as the abstaining set becomes so large that only very easy instances remain. Under PGD at $\epsilon=0.4$, selective accuracy is largely insensitive to $\alpha$, confirming that PGD at this budget primarily attacks the *confident* region, not the uncertain one.

- **$\beta$ (risk weight):** Primarily controls the FN/FP balance of committed decisions. Higher $\beta$ pushes the threshold toward abstaining on high-$p$ instances (where a wrong prediction would be a false negative), slightly increasing abstention. The effect on coverage is smaller than $\alpha$ or $c_\mathrm{abstain}$.

- **$\delta$ (information-utility weight):** The most interpretable trade-off. Higher $\delta$ incentivises committing to confident predictions, so coverage *rises* while selective accuracy on the committed set changes little. Too-high $\delta$ ($\ge 0.80$) drives coverage almost to 1, effectively recovering argmax behaviour.

- **$c_\mathrm{abstain}$ (abstention cost):** The dominant control knob for coverage. Small values → frequent abstention; large values → near-full commitment. This is the parameter to expose to end-users who want a simple coverage dial, holding everything else at defaults.

- **$w_\mathrm{FN}$ (FN/FP asymmetry):** Shifts the abstention boundary asymmetrically: the region near $p \approx 0.7$–$0.8$ becomes more likely to abstain as $w_\mathrm{FN}$ increases, reflecting the higher cost of a missed positive. Clinically relevant domains should set this $\ge 2$.

**Recommended defaults** (from the pareto analysis): $\alpha=0.30$, $\beta=0.50$, $\delta=0.40$, $c_\mathrm{abstain}=0.15$, $w_\mathrm{FN}=2.0$. These lie in the flat region of each 1D curve, i.e. performance is not sensitive to small perturbations around them. This is important for robustness of the configuration.

**Energy.**

- **$\alpha$ (task-loss weight):** Increasing $\alpha$ pulls the controller toward pure cost minimisation, reducing cost overhead to near zero but eliminating the stability benefit (SOC std converges to baseline). At $\alpha=0.50$ the controller is already well-regularised; at $\alpha=2.0$ the extra terms are drowned out.

- **$\beta$ (uncertainty weight):** Has a mild, monotonic effect: higher $\beta$ increases cost overhead by a few tenths of a percent but reduces SOC variance slightly. The $J$-reduction under worst-case FDI is largely unaffected ($\pm 2\%$ across the sweep), confirming that the uncertainty term is a calibration regulariser rather than the primary source of adversarial robustness.

- **$\gamma$ (risk weight):** The clearest energy trade-off. Higher $\gamma$ directly penalises overload and imbalance risk, reducing the rare high-overload events at the cost of higher routine operational cost (~0.5% per unit $\gamma$). The $J$-reduction under attack is stable across the $\gamma$ sweep, suggesting the risk term adds value primarily in the clean regime (reducing tail events) rather than under attack.

- **$\delta$ (information-utility / margin weight):** The only coefficient that materially improves the $J$-reduction under worst-case FDI. Higher $\delta$ rewards actions that preserve battery headroom, which makes the controller more conservative under manipulation: SOC is kept away from boundaries, reducing the attacker's ability to force expensive grid draws. The sweet spot is $\delta \approx 0.50$–$0.75$ before cost overhead climbs above 1%.

**Recommended defaults:** $\alpha=1.0$, $\beta=0.35$, $\gamma=0.60$, $\delta=0.50$. The dominant coefficient for adversarial robustness is $\delta$; the dominant coefficient for clean stability is $\gamma$. Practitioners who care primarily about attack resistance should raise $\delta$; those who care primarily about tail-event reduction should raise $\gamma$.